# Neutron-star EoS experiments

This notebook follows the physical workflow: choose an equation of state, inspect its domain and thermodynamics, integrate one TOV star, then calculate a mass–radius sequence.

> **Scientific boundary:** integrations stop at the lowest supplied positive pressure. Radii are therefore **source-boundary radii**, not vacuum-surface radii. The notebook does not repair inputs, splice crusts, extrapolate, infer stability, or calculate tidal observables.


## Before the first run

From the repository root, run `uv sync --all-extras`. To make a named Jupyter kernel, run `uv run python -m ipykernel install --user --name neutron-star-eos-toolkit --display-name "Python (neutron-star-eos-toolkit)"`, then select that kernel in VS Code or Jupyter.

The committed notebook uses the portable `python3` kernel name for automated testing. Use **Restart Kernel and Run All Cells** after changing parameters. The default experiment uses the bundled CSV and writes no files.


In [ ]:
from __future__ import annotations

import hashlib
import json
import runpy
import shutil
from dataclasses import asdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import JSON, Markdown, display

from neutron_star_eos import (
    EosInputError,
    EosModel,
    StellarConfig,
    StellarSolveError,
    open_eos,
)
from neutron_star_eos.plotting import (
    plot_compose_closure_residuals,
    plot_compose_cold_residuals,
    plot_compose_free_energy_closure_residuals,
    plot_composition,
    plot_mass_profile,
    plot_mass_radius,
    plot_phase_codes,
    plot_pressure_energy,
    plot_sequence_status,
    plot_sound_speed_squared,
)

## 1. Quick start: essential physical choices

For a first run, edit only these values. `INPUT_KIND` selects the source. The central pressure chooses one stellar model; the sequence controls choose a collection of models. Leave `ADVANCED_CONTROLS` as `None` until Section 6.


In [ ]:
# ---------------- parameters to edit first ----------------
REPOSITORY_ROOT = None  # Usually leave None; automatic repository search is used.
INPUT_KIND = "csv"  # "csv", "analytical", or "compose"
STAR_CENTRAL_PRESSURE_MEV_FM3 = 100.0
SEQUENCE_PRESSURE_RANGE_MEV_FM3 = None  # None samples the EoS pressure domain.
SEQUENCE_POINTS = 9
ADVANCED_CONTROLS = None  # Optional dictionary explained in Section 6.

### Input details

Edit only the block matching `INPUT_KIND`. For analytical work, the complete user function `P(epsilon)` and its consistent derivative live in `notebooks/analytical_eos.py`. CSV energy density must be total energy density including rest mass, in MeV/fm³. CompOSE stellar input must be cold, beta-equilibrated, and explicitly include leptons.


In [ ]:
# CSV input
CSV_PATH = "examples/tabulated.csv"
CSV_NAME = None
CSV_SOURCE_DESCRIPTION = None
CSV_EPSILON_COLUMN = "epsilon_mev_fm3"
CSV_PRESSURE_COLUMN = "pressure_mev_fm3"
CSV_BARYON_DENSITY_COLUMN = None

# Analytical input
ANALYTICAL_DEFINITION_PATH = "notebooks/analytical_eos.py"

# CompOSE input
COMPOSE_PATH = None
COMPOSE_MODEL_ID = None
COMPOSE_SOURCE_URL = None
COMPOSE_MATTER = "cold_beta_equilibrated"
COMPOSE_INCLUDES_LEPTONS = False
COMPOSE_BARYON_DENSITY_MIN_FM3 = None
COMPOSE_BARYON_DENSITY_MAX_FM3 = None
COMPOSE_NATIVE_POINTS = 2001
COMPOSE_ORDERING_POLICY = "strict"

In [ ]:
def find_repository_root(override=None):
    if override is not None:
        candidates = (Path(override).expanduser().resolve(),)
    else:
        start = Path.cwd().resolve()
        candidates = (start, *start.parents)
    for candidate in candidates:
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "src" / "neutron_star_eos"
        ).is_dir():
            return candidate
    raise RuntimeError(
        "Could not locate the repository. Set REPOSITORY_ROOT explicitly."
    )


REPO_ROOT = find_repository_root(REPOSITORY_ROOT)


def resolve_user_path(value):
    candidate = Path(value).expanduser()
    return (candidate if candidate.is_absolute() else REPO_ROOT / candidate).resolve()


def display_path(path):
    try:
        return path.relative_to(REPO_ROOT).as_posix()
    except ValueError:
        return str(path)


if INPUT_KIND not in {"csv", "analytical", "compose"}:
    raise ValueError("INPUT_KIND must be 'csv', 'analytical', or 'compose'")
if isinstance(SEQUENCE_POINTS, bool) or int(SEQUENCE_POINTS) < 2:
    raise ValueError("SEQUENCE_POINTS must be an integer of at least two")
if ADVANCED_CONTROLS is not None and not isinstance(ADVANCED_CONTROLS, dict):
    raise TypeError("ADVANCED_CONTROLS must be None or a dictionary")

advanced = {} if ADVANCED_CONTROLS is None else dict(ADVANCED_CONTROLS)
stellar_config = StellarConfig(
    radius_start_km=float(advanced.get("radius_start_km", 1.0e-4)),
    radius_max_km=float(advanced.get("radius_max_km", 50.0)),
    center_expansion_limit_km=float(advanced.get("center_expansion_limit_km", 1.0e-4)),
    ode_rtol=float(advanced.get("ode_rtol", 1.0e-10)),
    ode_atol=float(advanced.get("ode_atol", 1.0e-12)),
    profile_points=int(advanced.get("profile_points", 300)),
)
VALIDATION_MODE = str(advanced.get("validation_mode", "strict"))
ENABLE_BACKGROUND_DIAGNOSTIC = bool(advanced.get("enable_background_diagnostic", False))
RETAIN_STAR_PROFILE = bool(advanced.get("retain_star_profile", True))
PLOT_SAMPLE_POINTS = int(advanced.get("plot_sample_points", 513))

if VALIDATION_MODE not in {"strict", "background_diagnostic"}:
    raise ValueError("Unknown validation_mode in ADVANCED_CONTROLS")
if PLOT_SAMPLE_POINTS < 17:
    raise ValueError("plot_sample_points must be at least 17")

print("Repository:", REPO_ROOT)
print("Selected input kind:", INPUT_KIND)

## 2. Load the selected EoS and understand its domain

The analytical loader executes the editable definition file afresh and hashes it for provenance. CSV and CompOSE paths may be absolute or relative to the repository. No loader extrapolates beyond the declared source domain.


In [ ]:
def definition_hashes(path):
    raw = path.read_bytes()
    text = raw.decode("utf-8")
    canonical = text.replace("\r\n", "\n").replace("\r", "\n")
    return {
        "hash_policy": "utf8_text_normalized_lf_v1",
        "canonical_sha256": hashlib.sha256(canonical.encode("utf-8")).hexdigest(),
        "raw_sha256": hashlib.sha256(raw).hexdigest(),
    }


def load_analytical_model(path):
    before = definition_hashes(path)
    namespace = runpy.run_path(str(path), run_name="neutron_star_eos_user_definition")
    after = definition_hashes(path)
    if before != after:
        raise RuntimeError("The analytical definition changed while it was loading")
    build_model = namespace.get("build_model")
    if not callable(build_model):
        raise TypeError("analytical_eos.py must define callable build_model(...)")
    relative_path = display_path(path)
    description = str(
        namespace.get("SOURCE_DESCRIPTION", "User-defined analytical EoS")
    )
    source_identity = (
        f"{description}; definition={relative_path}; "
        f"normalized_lf_sha256={before['canonical_sha256']}"
    )
    loaded = build_model(source_identity=source_identity)
    if not isinstance(loaded, EosModel):
        raise TypeError("build_model(...) must return an EosModel")
    return loaded, {"repository_relative_path": relative_path, **before}

In [ ]:
definition_record = None
source_path = None

if INPUT_KIND == "csv":
    source_path = resolve_user_path(CSV_PATH)
    model = open_eos(
        source_path,
        kind="csv",
        name=CSV_NAME,
        source_description=CSV_SOURCE_DESCRIPTION,
        epsilon_column=CSV_EPSILON_COLUMN,
        pressure_column=CSV_PRESSURE_COLUMN,
        baryon_density_column=CSV_BARYON_DENSITY_COLUMN,
    )
elif INPUT_KIND == "analytical":
    source_path = resolve_user_path(ANALYTICAL_DEFINITION_PATH)
    model, definition_record = load_analytical_model(source_path)
else:
    if COMPOSE_PATH is None or COMPOSE_MODEL_ID is None or COMPOSE_SOURCE_URL is None:
        raise ValueError(
            "CompOSE input requires COMPOSE_PATH, COMPOSE_MODEL_ID, and "
            "COMPOSE_SOURCE_URL"
        )
    source_path = resolve_user_path(COMPOSE_PATH)
    model = open_eos(
        source_path,
        kind="compose",
        model_id=COMPOSE_MODEL_ID,
        source_url=COMPOSE_SOURCE_URL,
        matter=COMPOSE_MATTER,
        includes_leptons=COMPOSE_INCLUDES_LEPTONS,
        baryon_density_min_fm3=COMPOSE_BARYON_DENSITY_MIN_FM3,
        baryon_density_max_fm3=COMPOSE_BARYON_DENSITY_MAX_FM3,
        native_points=COMPOSE_NATIVE_POINTS,
        ordering_policy=COMPOSE_ORDERING_POLICY,
    )

print("Loaded source:", display_path(source_path))
if definition_record is not None:
    print("Analytical definition hash:", definition_record["canonical_sha256"])

In [ ]:
report = model.report()
print(model.summary())

eos = model.barotrope
if eos is None:
    display(Markdown("**No continuous stellar barotrope is available.**"))
else:
    print("\nDeclared continuous domain:")
    print(
        f"  energy density: {eos.energy_density_min_mev_fm3:.8g} to "
        f"{eos.energy_density_max_mev_fm3:.8g} MeV/fm^3"
    )
    print(
        f"  pressure:       {eos.pressure_min_mev_fm3:.8g} to "
        f"{eos.pressure_max_mev_fm3:.8g} MeV/fm^3"
    )
    print(
        "  lower boundary: finite positive source pressure; no P=0 surface is inferred"
    )

display(JSON(report.to_dict(), expanded=False))

## 3. Thermodynamic validation

Validation checks the continuous barotrope actually used by the TOV solver: positivity, monotonicity/invertibility, mechanical stability, and the causal guide `0 < c_s² ≤ 1`. CompOSE source diagnostics remain visible and are not silently repaired.


In [ ]:
figures = {}
thermodynamic_view = None

if eos is not None:
    validation = eos.validate()
    print("Continuous-barotrope validation:", "PASS" if validation.passed else "FAIL")
    print(f"Assessed points: {validation.assessed_points}")
    print(f"c_s^2 range: {validation.cs2_min:.8g} to {validation.cs2_max:.8g}")
    if validation.issues:
        print("Issues:", ", ".join(issue.code for issue in validation.issues))

thermodynamics_capability = report.capability("thermodynamics")
if thermodynamics_capability.available:
    thermodynamic_view = model.thermodynamics(curve_points=PLOT_SAMPLE_POINTS)
    for series in thermodynamic_view.series:
        print(
            f"{series.role}: {series.rows} rows; "
            f"columns={', '.join(series.column_names)}"
        )
else:
    display(Markdown(f"**Thermodynamics skipped:** {thermodynamics_capability.reason}"))

In [ ]:
if thermodynamic_view is not None:
    axis = plot_pressure_energy(
        model,
        curve_points=PLOT_SAMPLE_POINTS,
        show_source_nodes=True,
        show_stellar_barotrope="continuous_barotrope" in thermodynamic_view.roles,
    )
    axis.set_title(f"{model.model_name}: pressure–energy relation")
    figures["pressure_energy"] = axis.figure
    plt.show()

In [ ]:
if thermodynamic_view is not None:
    axis = plot_sound_speed_squared(
        model,
        curve_points=PLOT_SAMPLE_POINTS,
        include_stellar_barotrope="continuous_barotrope" in thermodynamic_view.roles,
    )
    axis.set_title(f"{model.model_name}: sound speed")
    figures["sound_speed_squared"] = axis.figure
    plt.show()

### Optional CompOSE diagnostics

These cells run only for CompOSE inputs. Closure residuals are diagnostics, composition keeps missing values as missing, and phase codes are categorical source information.


In [ ]:
if model.kind == "compose" and thermodynamic_view is not None:
    compose_plotters = {
        "compose_closure": plot_compose_closure_residuals,
        "compose_free_energy_closure": plot_compose_free_energy_closure_residuals,
        "compose_cold_residuals": plot_compose_cold_residuals,
    }
    for figure_name, plotter in compose_plotters.items():
        try:
            axis = plotter(model)
        except (EosInputError, KeyError, ValueError) as error:
            display(Markdown(f"**{figure_name} unavailable:** {error}"))
        else:
            figures[figure_name] = axis.figure
            plt.show()

    if report.capability("composition").available:
        axis = plot_composition(model)
        figures["composition"] = axis.figure
        plt.show()

    try:
        axis = plot_phase_codes(model)
    except (EosInputError, KeyError, ValueError) as error:
        display(Markdown(f"**Phase-code view unavailable:** {error}"))
    else:
        figures["phase_codes"] = axis.figure
        plt.show()

## 4. One-star TOV calculation

The central pressure from Section 1 fixes the central state. The toolkit then integrates the TOV equations outward until the lowest supplied positive pressure is reached. The exact numerical configuration is printed before integration.


In [ ]:
display(JSON(asdict(stellar_config), expanded=True))

stellar_capability = report.capability("stellar_background")
diagnostic_override = (
    not stellar_capability.available
    and VALIDATION_MODE == "background_diagnostic"
    and ENABLE_BACKGROUND_DIAGNOSTIC
    and model.barotrope is not None
)
stellar_authorized = stellar_capability.available or diagnostic_override
if diagnostic_override:
    display(
        Markdown(
            "**Diagnostic run enabled:** this does not upgrade EoS validation "
            "and must not be presented as a validated stellar result."
        )
    )

star = None
if stellar_authorized:
    try:
        star = model.solve_star(
            central_pressure_mev_fm3=STAR_CENTRAL_PRESSURE_MEV_FM3,
            config=stellar_config,
            retain_profile=RETAIN_STAR_PROFILE,
            validation_mode=VALIDATION_MODE,
        )
    except (EosInputError, StellarSolveError, ArithmeticError) as error:
        display(Markdown(f"**Star unavailable:** {error}"))
    else:
        print(f"Mass: {star.mass_msun:.8g} Msun")
        print(f"Source-boundary radius: {star.radius_km:.8g} km")
        print("Boundary status:", star.boundary_status)
        print("EoS validation status:", star.eos_validation_status)
else:
    display(Markdown(f"**Stellar calculation skipped:** {stellar_capability.reason}"))

In [ ]:
if star is not None and star.radius_profile_km:
    axis = plot_mass_profile(star)
    figures["mass_profile"] = axis.figure
    plt.show()

## 5. Mass–radius sequence

This is a requested collection of TOV integrations, not a stability analysis. Every requested pressure remains in the result, failed attempts keep their reasons, and no point is automatically called a maximum mass.


In [ ]:
sequence = None
if stellar_authorized:
    try:
        if SEQUENCE_PRESSURE_RANGE_MEV_FM3 is None:
            requested_pressures = None
        else:
            lower_pressure, upper_pressure = map(float, SEQUENCE_PRESSURE_RANGE_MEV_FM3)
            if not 0.0 < lower_pressure < upper_pressure:
                raise ValueError("Sequence range must satisfy 0 < lower < upper")
            requested_pressures = np.geomspace(
                lower_pressure, upper_pressure, int(SEQUENCE_POINTS)
            )
        sequence = model.solve_sequence(
            requested_pressures,
            points=SEQUENCE_POINTS,
            config=stellar_config,
            validation_mode=VALIDATION_MODE,
        )
    except (EosInputError, RuntimeError, ArithmeticError, ValueError) as error:
        display(Markdown(f"**Sequence unavailable:** {error}"))

if sequence is not None:
    print("Sequence status:", sequence.status)
    print("pressure [MeV/fm^3] | status      | mass [Msun] | radius [km] | reason")
    for attempt in sequence.attempts:
        if attempt.star is None:
            mass_text, radius_text = "-", "-"
        else:
            mass_text = f"{attempt.star.mass_msun:.8g}"
            radius_text = f"{attempt.star.radius_km:.8g}"
        print(
            f"{attempt.central_pressure_mev_fm3:20.8g} | "
            f"{attempt.status:11s} | {mass_text:11s} | {radius_text:11s} | "
            f"{attempt.reason or ''}"
        )

In [ ]:
if sequence is not None:
    axis = plot_mass_radius(
        sequence,
        connect=False,
        color_by_central_pressure=True,
    )
    axis.set_title(f"{model.model_name}: source-boundary backgrounds")
    figures["mass_radius"] = axis.figure
    plt.show()

In [ ]:
if sequence is not None:
    axis = plot_sequence_status(sequence)
    figures["sequence_status"] = axis.figure
    plt.show()

## 6. Optional advanced controls

Beginners should keep `ADVANCED_CONTROLS = None`. For a convergence study, copy the dictionary below into the quick-start parameter cell, edit one choice at a time, then **Restart Kernel and Run All Cells**. Background-diagnostic mode is not a physics-validation bypass.


In [ ]:
ADVANCED_CONTROLS_EXAMPLE = {
    "radius_start_km": 1.0e-4,
    "radius_max_km": 50.0,
    "center_expansion_limit_km": 1.0e-4,
    "ode_rtol": 1.0e-10,
    "ode_atol": 1.0e-12,
    "profile_points": 300,
    "plot_sample_points": 513,
    "validation_mode": "strict",
    "enable_background_diagnostic": False,
    "retain_star_profile": True,
}
print("Current numerical configuration:")
display(JSON(asdict(stellar_config), expanded=True))
print("Copy ADVANCED_CONTROLS_EXAMPLE into ADVANCED_CONTROLS only when needed.")

## 7. Optional saving and provenance

Saving is opt-in and refuses to overwrite an existing directory. Results include the model report, exact numerical parameters, source identity, figures, and—when analytical—the hashed definition file.


In [ ]:
# ---------------- optional output parameters ----------------
SAVE_RESULTS = False
OUTPUT_DIRECTORY = "notebook-output"
FIGURE_FORMATS = ("png", "svg")

In [ ]:
saved_directory = None
if SAVE_RESULTS:
    destination = resolve_user_path(OUTPUT_DIRECTORY)
    if sequence is not None:
        saved_directory = model.write_sequence(destination, sequence)
        saved_result_kind = "sequence"
    elif star is not None:
        saved_directory = model.write_star(destination, star)
        saved_result_kind = "star"
    else:
        saved_directory = model.write_inspection(destination)
        saved_result_kind = "inspection"

    figure_directory = saved_directory / "figures"
    figure_directory.mkdir()
    for figure_name, figure in figures.items():
        for suffix in FIGURE_FORMATS:
            normalized_suffix = str(suffix).lower().lstrip(".")
            if normalized_suffix not in {"png", "svg", "pdf"}:
                raise ValueError(f"Unsupported figure format: {suffix}")
            save_options = {"dpi": 300} if normalized_suffix == "png" else {}
            figure.savefig(
                figure_directory / f"{figure_name}.{normalized_suffix}",
                **save_options,
            )

    copied_definition = None
    if definition_record is not None:
        copied_path = saved_directory / "analytical_eos.py"
        shutil.copy2(source_path, copied_path)
        copied_definition = {
            "copied_filename": copied_path.name,
            **definition_hashes(copied_path),
        }
        if (
            copied_definition["canonical_sha256"]
            != definition_record["canonical_sha256"]
        ):
            raise RuntimeError("Copied analytical definition failed hash verification")

    manifest = {
        "schema_version": "eos-notebook-experiment-v1",
        "input_kind": model.kind,
        "result_kind": saved_result_kind,
        "source_path": display_path(source_path),
        "analytical_definition": definition_record,
        "copied_analytical_definition": copied_definition,
        "parameters": {
            "validation_mode": VALIDATION_MODE,
            "star_central_pressure_mev_fm3": STAR_CENTRAL_PRESSURE_MEV_FM3,
            "sequence_pressure_range_mev_fm3": SEQUENCE_PRESSURE_RANGE_MEV_FM3,
            "sequence_points": SEQUENCE_POINTS,
            "stellar_config": asdict(stellar_config),
            "plot_sample_points": PLOT_SAMPLE_POINTS,
        },
        "model_report": report.to_dict(),
    }
    (saved_directory / "experiment.json").write_text(
        json.dumps(manifest, indent=2, sort_keys=True, allow_nan=False) + "\n",
        encoding="utf-8",
        newline="\n",
    )
    print("Saved experiment:", saved_directory)
else:
    print("SAVE_RESULTS is False; no files were written.")

In [ ]:
default_csv_run = (
    INPUT_KIND == "csv"
    and source_path == (REPO_ROOT / "examples" / "tabulated.csv").resolve()
    and STAR_CENTRAL_PRESSURE_MEV_FM3 == 100.0
    and SEQUENCE_PRESSURE_RANGE_MEV_FM3 is None
    and SEQUENCE_POINTS == 9
    and stellar_config.radius_max_km == 50.0
)
if default_csv_run:
    assert thermodynamic_view is not None
    assert star is not None
    assert sequence is not None and sequence.status == "complete"
    assert len(sequence.attempts) == SEQUENCE_POINTS

print("EOS_NOTEBOOK_EXECUTION_OK")